# Week 9: Relational Algebra + SQL (2)

## Background

Before moving to SQL, we will review how to combine and summarize relational data using pandas. These examples connect relational algebra concepts to practical data analysis and prepare us for similar operations in SQL.

## Lecture Outline

- Filtering data and joining tables
- Combining real-world NYC flight datasets
- Grouping and aggregating data
- Filtering groups and selecting records
- Creating pivot tables

## Required Packages

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import pprint

import requests
import json
from pathlib import Path
import yaml
from datetime import datetime

## Example: Superhero Tables

We begin with three small tables describing superheroes, their identities, and their teams. We will use these tables to demonstrate how shared keys connect related records.

*Example adapted from Data100.*

The superheroes come from the Arrowverse. In the `first_seen_on` column, `a` represents *Arrow* and `f` represents *The Flash*.

In [ ]:
heroes = pd.DataFrame(
    data={'color': ['red', 'green', 'black', 
                    'blue', 'black', 'red'],
          'first_seen_on': ['a', 'a', 'f', 'a', 'a', 'f'],
          'first_season': [2, 1, 2, 3, 3, 1]},
    index=['flash', 'arrow', 'vibe', 
           'atom', 'canary', 'firestorm']
)

identities = pd.DataFrame(
    data={'ego': ['barry allen', 'oliver queen', 'cisco ramon',
                  'ray palmer', 'sara lance', 
                  'martin stein', 'ronnie raymond'],
          'alter-ego': ['flash', 'arrow', 'vibe', 'atom',
                        'canary', 'firestorm', 'firestorm']}
)

teams = pd.DataFrame(
    data={'team': ['flash', 'arrow', 'flash', 'legends', 
                   'flash', 'legends', 'arrow'],
          'hero': ['flash', 'arrow', 'vibe', 'atom', 
                   'killer frost', 'firestorm', 'speedy']})

In [ ]:
heroes

In [ ]:
identities

In [ ]:
teams

### `IsIn` 

An often used operation is a test-of-membership.  The `Series.isin(values)` method returns a boolean array denoting whether each element of `Series` is in `values`.  We can then use the array to subset our DataFrame.  For example, if we wanted to see which rows of `heroes` had `first_season` values in $\{1, 3\}$, we would use: 

In [ ]:
heroes[heroes['first_season'].isin([1, 3])]

## Joining Tables on One Column 

Suppose we have another table, `teams`, that classifies superheroes into their respective teams.  Note, that `canary` is not in the `teams` data set and that `killer frost` and `speedy` are not in the original `heroes` set.

For simplicity of the example, we'll convert the index of the `heroes` DataFrame into an explicit column called `hero`.  A careful examination of the `merge` [documentation](http://pandas.pydata.org/pandas-docs/version/0.19.1/generated/pandas.DataFrame.merge.html) will reveal that joining on a mixture of the index and columns is possible. 

In [ ]:
heroes['hero'] = heroes.index
heroes

In [ ]:
teams

To understand the different types of join or `merge` in `pandas`, we will use the examples given: 

* Natural join: Keep only rows that match from the data frames, `how='inner'` 
* Full outer join: Keep all rows taht match from both data frames `how='outer'` 
* Left outer join: Include all rows of your data frame `x` and only those from `y` that match, `how='left'` 
* Right outer join: Include all rows of your data frame `y` and only those from `x` that match `how='right'`

<img src="https://pages.mtu.edu/~lebrown/un5550-f20/week8/join-or-merge-in-pandas.png">

### Inner Join 

The inner join returns rows representing the heroes that appear in both data frames. 

In [ ]:
pd.merge(heroes, teams, how="inner", on="hero")

### Left Join 

The left join returns rows reprsenting `heroes` in the "left" data frame augmented by information found in the `teams` data frame.  

Note, the `team` value for `canary` is `NaN`, representing missing value.*italicized text*

In [ ]:
pd.merge(heroes, teams, how="left", on="hero")

The right join returns the `hero` values in the `teams` data frame.

In [ ]:
pd.merge(heroes, teams, how="right", on="hero")

### Outer Join 

An outer join on `hero` will return all heroes found in both the left and right data frames.  Any missing values are filled in with `NaN`

In [ ]:
pd.merge(heroes, teams, how="outer", on="hero")

Note, the join operations can also be run using the following notation. 

In [ ]:
heroes.merge(teams, how="outer", on="hero")

## Example: NYC Flights, Airlines, and Aircraft

We will combine three real-world CSV tables from **nycflights13**, containing flights departing New York City in 2013:

* `planes.csv`: aircraft information; one row per `tailnum` (aircraft registration).
* `airlines.csv`: airline names; one row per `carrier` code.
* `flights.csv`: individual departures, linking aircraft to airlines through `tailnum` and `carrier`.

An aircraft can appear on many flights, and an airline operates many flights. Unlike the original legislator example, these links represent **events**, so repeated aircraft–airline pairs are meaningful.

Place all three CSV files in the same folder as this notebook. These are real records, not simulated data. The flight CSV retains eight columns from the original table; no flight rows are removed.

Source: [nycflights13 documentation](https://nycflights13.tidyverse.org/), distributed as CSV through [Rdatasets](https://vincentarelbundock.github.io/Rdatasets/). Aircraft metadata originates from the FAA; flight records originate from the US Bureau of Transportation Statistics. Aircraft age below is measured approximately in **2013**, not today.

In [ ]:
planes = pd.read_csv('planes.csv')
planes.head()

First, inspect the aircraft table. The `tailnum` column identifies each aircraft; `year` is its manufacturing year.

In [ ]:
len(planes)

In [ ]:
planes['tailnum'].is_unique

In [ ]:
plane_df = planes[['tailnum', 'year', 'manufacturer', 'model']].copy()
plane_df['age'] = 2013 - plane_df['year']
plane_df.head()

### Load Airline and Flight Data

Next, load the airline lookup table and the flight records that connect aircraft to airlines.

In [ ]:
airline_df = pd.read_csv('airlines.csv')
flight_df = pd.read_csv('flights.csv')
print(len(airline_df), len(flight_df))

Inspect the airline descriptions. Each carrier code maps to one airline name.

In [ ]:
airline_df.head()

### Inspect the Linking Table

Each row in `flight_df` is one flight. The `carrier` key links to airlines, and `tailnum` links to aircraft. A repeated pair represents another flight, not an accidental duplicate.

In [ ]:
flight_df.head()

The `carrier` column tells us which airline operated each flight.

In [ ]:
flight_df['carrier'].unique()

Look at a few departures operated by United Air Lines (`UA`).

In [ ]:
flight_df[flight_df['carrier'] == 'UA'].head()

In [ ]:
flight_df[flight_df['carrier'] == 'UA'].iloc[0]

### Select the Columns Linking Flights to Aircraft and Airlines

Keep the two join keys. Repeated rows are retained because each row still represents a separate departure.

In [ ]:
flight_links = flight_df[['carrier', 'tailnum']].copy()
flight_links.head()

### Merge DataFrames

Join the flight links to airline names. This is a many-to-one join: many flights share one airline code. An inner join retains flights with a matching airline record.

In [ ]:
flight_airlines = flight_links.merge(airline_df, on='carrier', how='inner', validate='many_to_one')
flight_airlines.head()

In [ ]:
flight_airlines2 = pd.merge(flight_links, airline_df, on='carrier', how='inner', validate='many_to_one')
flight_airlines2.head()

In [ ]:
airline_sizes = flight_airlines['name'].value_counts()
airline_sizes.head()

Now join the combined table to aircraft metadata using `tailnum`. The inner join excludes flights whose aircraft registration is missing or not found in `planes.csv`. Thus, the next summaries describe matched aircraft only.

In [ ]:
flight_details = flight_airlines.merge(plane_df, on='tailnum', how='inner', validate='many_to_one')
flight_details.head()

What is the mean aircraft age for each airline? Each row is a flight, so aircraft used more often receive more weight. Missing manufacturing years are omitted by `mean()`.

In [ ]:
airline_ages = flight_details['age'].groupby(flight_details['name']).mean().sort_values()
airline_ages.head()

For comparison, calculate the mean age across the aircraft metadata table. Here each aircraft has equal weight, and missing ages are excluded. This is a different population and weighting from the flight-level summary above.

In [ ]:
plane_df['age'].mean()

### Calculate the Number of Flights for Each Aircraft

Count departures per aircraft. `value_counts()` excludes missing registrations by default. These are flights from NYC in this dataset, not each aircraft’s worldwide annual flight total.

In [ ]:
flight_count = flight_links['tailnum'].value_counts().to_frame().reset_index()
flight_count.columns = ['tailnum', 'num_flights']
flight_count.head()

Merge the counts with the aircraft table to include manufacturing year, model, and age. The inner join retains aircraft that have both metadata and recorded flights.

In [ ]:
aircraft_usage = plane_df.merge(flight_count, on='tailnum', how='inner', validate='one_to_one')
aircraft_usage.head()

Finally, plot aircraft age against the number of NYC departures. Each point represents one aircraft. Rows with unknown age are excluded explicitly. The fitted line describes association; it does not establish causation.

In [ ]:
sns.lmplot(x='age', y='num_flights', data=aircraft_usage.dropna(subset=['age']), fit_reg=True);
plt.xlabel('Approximate aircraft age in 2013 (years)')
plt.ylabel('Number of NYC departures in 2013')
plt.title('Aircraft Age and NYC Flight Count');

### Summary

We used a linking table to join three CSV datasets, counted records by airline, computed mean aircraft age by airline, counted flights per aircraft, merged those counts back to aircraft metadata, and plotted age against usage.

**Quick check:** Why can the mean aircraft age calculated from flight records differ from the mean age in the aircraft table?

In [ ]:
aircraft_usage[['age', 'num_flights']].describe()

# Review of Grouping Data 

For most of the course so far, we have been working with a single data file at a time. 

Let's look at how we can do grouping within a single data set and also working across multiple data sets.

In [ ]:
df = pd.read_csv("elections.csv")
df.head()

## Grouping a Series by a Series 

Let's group the 'Perc' Series by the `Party` Series. We have seen a call to [`groupby`](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.groupby.html) do this. 

In [ ]:
percent_grouped_by_party = df['Perc'].groupby(df['Party'])
type(percent_grouped_by_party)

As we see, `percent_grouped_by_party` is **NOT** a DataFrame, Series, or Index. Instead, it's a `SeriesGroupBy` object. A `SeriesGroupBy` consists of `groups`, one for each of the distinct values of the `Party` column. If we ask to see these groups, we'll be able to see which indices in the original DataFrame correspond to each group.

In [ ]:
#import pprint
pprint.pprint(percent_grouped_by_party.groups)

The `percent_grouped_by_party` object is capable of making computations across all these groups.  

For example, if we call the `mean` method of the `SeriesGroupBy` class.  We'll get a new `Series` containing the mean of the "Democratic" `Series`, the mean of the "Independent" `Series`, and the mean of the "Republican" `Series`.

In [ ]:
percent_grouped_by_party.mean()

We can iterate over a `SeriesGroupBy` object.  Each element is a pair of `(name, group)`, where `name` is a String label for the group and `group` is a `Series` corresponding to all the values from that group. 

In [ ]:
from IPython.display import display  # like print, but for complex objects

for name, group in percent_grouped_by_party:
    print('Name:', name)
    print(type(group))
    display(group.head())
    print('\n')

## Grouping a Series by Multiple Series 

We can also group a Series by multiple Series.  For example, suppose we want to track all combinations of `{'Democratic', 'Republican', and 'Independent'}` and `{'win', 'loss'}`. 

In [ ]:
percent_grouped_by_party_and_result = df['Perc'].groupby([df['Party'], df['Result']])
percent_grouped_by_party_and_result.groups

In [ ]:
pprint.pprint(percent_grouped_by_party_and_result.groups)

Given this groupby object, we can compute the average percentage earned every time each of the parties won and lost the presidential election. We see that at least between 1980 and 2016, the Republicans have typically lost and won their elections by wider margins.

In [ ]:
party_result_mean = percent_grouped_by_party_and_result.mean()
party_result_mean

The careful reader will note that the returned object looks a little funny. It seems to have two indexes! If we check the type of this object, we'll see it's just a regular Series.

In [ ]:
type(party_result_mean)

However if we request to see the index of this Series, we see that it is a "MultiIndex", which is a special type of index used for data that is indexed in two or more ways.

In [ ]:
party_result_mean.index

Selecting an element of a multi-index gives you a series with just an index.

In [ ]:
party_result_mean['Republican']

Or, you can select both levels at once. 

In [ ]:
party_result_mean['Democratic', 'win']

## Grouping a DataFrame by a Series

We can also group an entire dataframe by one or more Series. The result is a DataFrameGroupBy object:

In [ ]:
everything_grouped_by_party = df.groupby('Party')
everything_grouped_by_party

As in our previous example, this object contains three group objects, one for each party label. 

In [ ]:
pprint.pprint(everything_grouped_by_party.groups)

Just as with SeriesGroupBy objects, we can iterate over a DataFrameGroupBy object to understand what is effectively inside.

In [ ]:
for n, g in everything_grouped_by_party:
    print('name:', n)
    display(g.head())

And just like SeriesGroupBy objects, we can apply methods like mean to compute the mean of each group. Since a DataFrameGroupBy is linked to the entire original dataframe (instead of to a single column from the dataframe), we calculate a mean for every numerical column. 

In [ ]:
everything_grouped_by_party[['Perc', 'Year']].mean()

## Grouping a DataFrame by MultipleSeries

DataFrames may also be grouped by multiple series at once. For example, we can repeat what we did with a Series above and group the entire DataFrame by Party and Result. After aggregation, we end up with a DataFrame that has a MultiIndex.

In [ ]:
everything_grouped_by_party_and_result=df.groupby([df['Party'], df['Result']])
everything_grouped_by_party_and_result.max()

The resulting DataFrame above is pretty strange. We'll observe that Walter Mondale did not run for office in 2016! Make sure you understand why this is happening, as this sort of thing is a common mistake made by people who don't fully understand how pandas works.

**Question:** How do you generate a table like the one above, except that each entry should represent all attributes of the candidate who got the maximum vote in that category? For example, the Republican Win category should feature Reagan in 1984 with 58.8% of the vote. *Hint:* Try sorting.

In [ ]:
(df.sort_values("Perc", ascending=False).
    groupby([df['Party'], df['Result']]).
    first())

## Custom Aggregation Functions 

As described above, both `SeriesGroupBy` and `DataFrameGroupBy` objects have lots of handy methods for computing aggregate values for groups, e.g.

In [ ]:
percent_grouped_by_party.min()

In [ ]:
everything_grouped_by_party['Perc'].median()

It turns out that all of these GroupBy methods are just shorthand for a more powerful and universal method of our GroupBy objects called `agg`. For example, `.min()` is just shorthand for `.agg(min)`, where `min` refers to the function `min`.

In [ ]:
percent_grouped_by_party.agg(min)

Naturally, we can define our own custom aggregation functions. For example, the function below returns the first item in a series.

In [ ]:
def average_of_first_and_last(series):
    return (series.iloc[0] + series.iloc[-1])/2

We can supply this function as a custom aggregation function for each series. As you can see, nuisance columns are automatically removed.

In [ ]:
percent_grouped_by_party_and_result.agg(average_of_first_and_last)

`agg` is fundamental to our use of GroupBy objects. Indeed, you will rarely call `groupby` without also calling `agg`, at least implicitly. We can summarize the grouping process with the following diagram, inspired by a similar diagram created by DS100 instructor Joey Gonzales. Diagram source at [this link](https://docs.google.com/presentation/d/1FrYg6yd6B-CIgfWLWm4W8vBhfmJ6Qt9dKkN-mlN5AKU/edit#slide=id.g4131093782_0_40).

<img src="https://pages.mtu.edu/~lebrown/un5550-f20/week8/groupby_overview.png">

The result of calling `groupby` then `agg` on a `Series` is also a `Series`, and the result of calling `groupby` then `agg` on a `DataFrame` is also typically a `DataFrame`, though there are exceptions, e.g. if you use the aggregation function `size`, you will get back a `Series`.

In [ ]:
def list_of_first_and_last(series):
    return [series.iloc[0], series.iloc[-1]]

In [ ]:
everything_grouped_by_party.agg(list_of_first_and_last)

## Using Groups to Filter Datasets 

In [ ]:
everything_grouped_by_candidate = df.groupby('Candidate')
everything_grouped_by_candidate.filter(lambda subframe: subframe["Year"].max() > 2000)

In [ ]:
everything_grouped_by_year = df.groupby('Year')

In [ ]:
everything_grouped_by_year.filter(lambda subframe: subframe["Perc"].sum() < 97)

A visual picture of how filtering works is shown below.

<img src="https://pages.mtu.edu/~lebrown/un5550-f20/week8/filter_overview.png">

## Using `isin` for filtering

This isn't actually related to `groupby` at all, but it seemed natural to bring it up here.

Sometimes we want to restrict our attention to only rows where certain values appear. For example, we saw last time how we'd look at only rows that contain "Democratic" candidates.

In [ ]:
df[df["Party"] == "Democratic"]

Suppose we wanted to filter such that all Republicans and Democrats appeared. One ugly way to do this would be `df[(df["Party"] == "Democratic") | (df["Party"] == "Republican")]`. However, a better idea is to use the `isin` method.

In [ ]:
df[(df["Party"] == "Democratic") | (df["Party"] == "Republican")]


In [ ]:
df[df["Party"].isin(["Republican", "Democratic"])]

## Pivot Tables 

Recall from before that we were able to group the "Perc" Series by the "Party" and "Result" Series, allowing us to understand the average vote earned by each party under each election result.

In [ ]:
percent_grouped_by_party_and_result = df['Perc'].groupby([df['Party'], df['Result']])
percent_grouped_by_party_and_result.mean()

Because we called `groupby` on a Series, the result of our aggregation operation was also a Series. However, I believe this data is more naturally expressed in a tabular format, with Party as the rows, and Result as the columns. The `pivot_table` operation is the natural way to achieve this data format.

In [ ]:
df_pivot = df.pivot_table(
    index='Party', # the rows (turned into index)
    columns='Result', # the column values
    values='Perc', # the field(s) to processed in each group
    aggfunc=np.mean, # group operation
)
df_pivot.head()

The basic idea is that you specify a Series to be the `index` (i.e. rows) and a Series to be the `columns`. The data in the specified `values` is then grouped by all possible combinations of values that occur in the `index` and `columns` Series. These groups are then aggregated using the `aggfunc`, and arranged into a table that matches the requested `index` and `columns`. The diagram below summarizes how pivot tables are formed. (Diagram inspired by Joey Gonzales). Diagram source at [this link](https://docs.google.com/presentation/d/1FrYg6yd6B-CIgfWLWm4W8vBhfmJ6Qt9dKkN-mlN5AKU/edit#slide=id.g4131093782_0_89).

<img src="https://pages.mtu.edu/~lebrown/un5550-f20/week8/pivot_table_overview.png">

For more on pivot tables, see [this excellent tutorial](http://pbpython.com/pandas-pivot-table-explained.html) by Chris Moffitt.

# See you next lecture!